# BFS 與 DFS 入門講義

> 目標：從生活情境出發理解圖形(Graph)走訪的兩大策略 —— **BFS** 與 **DFS**，掌握資料結構選擇、走訪順序與典型應用，最後透過小練習加深印象。


## 0. 為什麼需要 BFS / DFS？
- 在迷宮、社群網路、路線規劃等問題中，我們常需要從一個節點出發探索「能到達哪些節點」。
- **BFS**（Breadth-First Search）：像是「一圈一圈擴散」的方式，適合找最少步數或最短距離。
- **DFS**（Depth-First Search）：像是「沿著一條路走到底再回頭」，適合確認是否存在路徑、計算連通區塊大小、找所有可能組合。

> 本講義會先整理共同的前置觀念，再分別深入 BFS 與 DFS 的流程與練習。


## 1. 圖形與鄰居的基本概念
- 我們用「節點 (node)」與「邊 (edge)」來描述資料的關聯。
- 在程式中常用「鄰居列表」(adjacency list) 或「偏移列表」(dir offsets) 來取得某節點的鄰居。
- 走訪圖形時要注意：
  1. 如何取得鄰居？
  2. 如何避免重複造訪？（需要 `visited` 結構）
  3. 何時停止走訪？（找到目標或全部節點）


In [119]:
# 一個簡易的鄰居列表示例
neighbors = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': ['C', 'E']
}

for node, adj in neighbors.items():
    print(f"{node} -> {adj}")


A -> ['B', 'C']
B -> ['A', 'D', 'E']
C -> ['A', 'F']
D -> ['B']
E -> ['B', 'F']
F -> ['C', 'E']


## 2. BFS：層層推進的走訪
**核心想法**：
1. 使用佇列 (queue)，先進先出。
2. 先把起點放進佇列，並標記為已訪問。
3. 每次從佇列取出一個節點，依序將尚未拜訪的鄰居加入佇列。
4. 重複直到佇列為空或達到目標。

### 2.1 BFS 手動演練
以下程式示範從節點 `'A'` 開始的 BFS 順序。


In [120]:
from collections import deque

neighbors = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': ['C', 'E']
}

queue = deque(['A'])
visited = {'A'}
order = []

while queue:
    node = queue.popleft()
    order.append(node)
    for nei in neighbors[node]:
        if nei not in visited:
            visited.add(nei)
            queue.append(nei)

print('BFS 順序:', order)
print('拜訪順序追蹤:', order)


BFS 順序: ['A', 'B', 'C', 'D', 'E', 'F']
拜訪順序追蹤: ['A', 'B', 'C', 'D', 'E', 'F']


### 2.2 BFS 找最短步數
- BFS 的特性是在無權重圖中找出最短步數。
- 下例使用 `dist` 字典記錄從起點 `'A'` 到每個節點的距離。


In [121]:
from collections import deque

start = 'A'
dist = {start: 0}
queue = deque([start])

while queue:
    node = queue.popleft()
    for nei in neighbors[node]:
        if nei not in dist:
            dist[nei] = dist[node] + 1
            queue.append(nei)

print('各節點距離:')
for node in sorted(dist):
    print(f"{node}: {dist[node]}")


各節點距離:
A: 0
B: 1
C: 1
D: 2
E: 2
F: 2


### 2.3 BFS 練習：圖形走訪
1. 修改鄰居列表，加入節點 `G` 與對應邊，觀察 BFS 順序如何改變。
2. 實作函式 `bfs_path(graph, start, target)`，回傳從 `start` 到 `target` 的最短路徑（節點列表）。
3. 試著將 BFS 應用到 2D 迷宮（可參考 `grid_neighbor_counts.ipynb` 中的示例）。


In [122]:
# TODO：實作 BFS 練習
# 1. 擴充 neighbors，或自行設計一個圖形
# 2. 完成 bfs_path 函式
# 3. 測試從 'A' 到 'F' 是否能找到最短路徑


## 3. DFS：走到底再回頭
**核心想法**：
1. 遞迴或使用堆疊 (stack) 實作。
2. 選一個未訪問鄰居深入，直到沒有路可以走，再回溯。
3. 常用於驗證路徑存在、列舉所有組合、計算連通區塊。 

### 3.1 遞迴版 DFS
以下範例列出從 `'A'` 出發的 DFS 拜訪順序。


In [123]:
visited = set()
order = []

def dfs(node):
    visited.add(node)
    order.append(node)
    for nei in neighbors[node]:
        if nei not in visited:
            dfs(nei)

dfs('A')
print('DFS 順序:', order)


DFS 順序: ['A', 'B', 'D', 'E', 'F', 'C']


### 3.2 stack 版 DFS
- 若不想用遞迴，可使用自建堆疊模擬呼叫堆疊。
- 注意：為了與遞迴一致，需要視情況調整鄰居的加入順序。


In [124]:
stack = ['A']
visited = set()
order = []

while stack:
    node = stack.pop()
    if node in visited:
        continue
    visited.add(node)
    order.append(node)
    # 為了與遞迴順序類似，需逆序放入鄰居
    for nei in reversed(neighbors[node]):
        if nei not in visited:
            stack.append(nei)

print('Stack 版 DFS 順序:', order)


Stack 版 DFS 順序: ['A', 'B', 'D', 'E', 'F', 'C']


### 3.3 DFS 練習
1. 修改 DFS 讓它回傳「從起點到所有節點的父節點映射」，用於回溯路徑。
2. 嘗試在圖中加入環（例如讓 `F` 連回 `A`），確認 visited 機制能避免無限遞迴。
3. 觀察不同鄰居排序對 DFS 拜訪順序的影響。


In [125]:
# TODO：完成 DFS 練習
# - 建立 parent 字典
# - 印出從 'A' 到其他節點的 DFS 路徑


## 4. BFS vs DFS：比較表
| 面向 | BFS | DFS |
| --- | --- | --- |
| 使用資料結構 | 佇列 (queue) | 堆疊 (stack) 或遞迴 |
| 典型用途 | 最短路徑、最少步數 | 連通區塊、存在性檢查、枚舉所有解 |
| 走訪順序 | 逐層擴散 | 沿一條路走到底再回頭 |
| 是否一定找到最短路徑 | ✅（無權重圖） | ❌（除非探索所有路徑並比較） |
| 實作風格 | 通常使用迴圈 | 遞迴或迴圈皆可 |

> 小提醒：兩者都需要 `visited`，用以避免重複拜訪。


## 5. 綜合練習
1. 自行設計一張圖（節點至少 6 個），分別用 BFS 與 DFS 列出拜訪順序。
2. 為該圖寫出 `bfs_path` 與 `dfs_path` 函式，回傳從起點到終點的節點序列，並比較差異。
3. 將 BFS/DFS 應用到 2D 網格：
   - BFS 找出最短路徑。
   - DFS 計算連通通道的格子數。
4. 思考題：如果圖有權重，BFS 還能保證最短路徑嗎？為什麼？（提示：需要 Dijkstra 或其他演算法。）

> 建議先親手筆記 BFS/DFS 的步驟，再實際執行程式驗證。
